[//]: # (cr:doc name='probe_overview' id=probe-intro)
# Sanity-Check Probe (universal template)

Parametrized probe for validating landing / bronze / silver artifacts of
a CustomerRetention run against a declared dataset contract.

**This template is client-agnostic.** Instantiate it for an engagement by:
1. Copying to `debug/<engagement>/probe.ipynb`.
2. Filling the `DATASETS` / `CONTRACTS` dicts in cell `probe-config`.
3. Running against a `RUN_ID`.

The probe never imports `pandas as pd`, never swallows errors, and stays
distributed (pyspark.pandas / pyspark.sql) — see `docs/Coding_Practices.md`.


In [ ]:
# @cr:config name='probe_config' id=probe-config
RUN_ID = ""
EXPERIMENTS_SUBDIR = "experiments"

# Absolute path to experiments root (Volumes on Databricks).
# Fill this in; framework auto-resolution can pick the wrong path
# on Workspace-Repos clusters.
EXPERIMENTS_ROOT = "<set to /Volumes/<catalog>/<schema>/experiments>"

DATASETS: dict[str, dict] = {
    # "<name>": {
    #     "kind": "entity" | "event",
    #     "entity_cols": ["ACCOUNT_ID"],
    #     "time_col": "CREATED_DATE",
    #     "bronze_path": "bronze/<name>" | "bronze/<name>_aggregated",
    #     "per_grid_date_mode": bool,
    #     "has_lifecycle_enrichment": bool,
    #     "event_types": [...],
    #     "value_column": str | None,
    #     "landing_required_cols": [...],
    #     "landing_forbidden_cols": [...],
    #     "silver_prefix": "<name>__",
    # },
}

BRONZE_WINDOWS: list[str] = []

FORBIDDEN_SILVER_SUBSTRINGS: list[str] = []

EXPECTED_MILESTONE_COLUMNS: dict[str, list[str]] = {}

TARGET_HOST_DATASET: str | None = None
TARGET_COLUMN: str = "churned"


In [ ]:
# @cr:code name='init_progress' id=probe-init
from customer_retention.analysis.notebook_progress import accept_workflow_params

accept_workflow_params()

import json
from pathlib import Path

from pyspark.sql import functions as F

from customer_retention.analysis.auto_explorer import RunNamespace, mark_notebook
from customer_retention.analysis.visualization import console, display_table
from customer_retention.core.compat import native_pd

_EXPERIMENTS_PATH = Path(EXPERIMENTS_ROOT)
if RUN_ID:
    _namespace = RunNamespace(root=_EXPERIMENTS_PATH, run_id=RUN_ID)
else:
    _namespace = RunNamespace.from_env_or_latest(root=_EXPERIMENTS_PATH)
mark_notebook(_namespace, "probe.ipynb")

RUN_ROOT = str(_EXPERIMENTS_PATH / "runs" / _namespace.run_id / "data")
RESULT_DIR = Path(_namespace.session_dir) / "probe"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

CHECKS: list[dict] = []

# --- cr:profiler ---
if __import__('os').environ.get("CR_BATCH_EXECUTION") == "1":
    import json as _j
    import os as _os
    import re as _r
    _cr_nb = _os.path.splitext(_os.path.basename(_os.environ.get("PAPERMILL_OUTPUT_PATH", "")))[0]
    if _cr_nb:
        _cr_mp = _os.path.join(_os.getcwd(), f".cr_cell_metrics_{_cr_nb}.jsonl")
        open(_cr_mp, 'w').close()
        _cr_re = _r.compile(r"^#\s*@cr:\w+\s+name='([^']+)'\s+id=(\w+)")
        def _cr_jc():
            return -1
        try:
            _s = __import__('pyspark.sql', fromlist=['SparkSession']).SparkSession.getActiveSession()
            if _s:
                def _cr_jc():  # noqa: F811
                    return _s._jsc.sc().dagScheduler().nextJobId().get()
        except Exception:
            pass
        def _cr_pre(info):
            info._cr_sj = _cr_jc()
        def _cr_post(r):
            sj = getattr(r.info, '_cr_sj', -1)
            sa = _cr_jc()
            m = _cr_re.match((r.info.raw_cell or '').split('\n')[0])
            if m:
                with open(_cr_mp, 'a') as f:
                    f.write(_j.dumps({"cell_name": m.group(1), "cell_id": m.group(2),
                                      "spark_jobs": (sa - sj) if sj >= 0 and sa >= 0 else None}) + '\n')
        get_ipython().events.register('pre_run_cell', _cr_pre)
        get_ipython().events.register('post_run_cell', _cr_post)
# --- /cr:profiler ---


In [ ]:
# @cr:code name='probe_helpers' id=probe-helpers
def _try_load(path):
    try:
        df = spark.read.format("delta").load(path)
        _ = df.columns
        return df, None
    except Exception as e:
        return None, f"{type(e).__name__}: {str(e)[:200]}"


def _record(layer, dataset, check, passed, detail=""):
    CHECKS.append({
        "layer": layer, "dataset": dataset, "check": check,
        "status": "PASS" if passed else "FAIL", "detail": str(detail)[:200],
    })


def _first_col(df, candidates):
    return next((c for c in candidates if c in df.columns), None)


def _section(title):
    print(f"\n{'='*72}\n  {title}\n{'='*72}")


def _dump(rows, label):
    if not rows:
        return None
    native_pd.DataFrame(rows).to_csv(RESULT_DIR / f"{label}.csv", index=False)
    return str(RESULT_DIR / f"{label}.csv")


[//]: # (cr:doc name='layer_1_landing' id=probe-l1-md)
## Layer 1 — Landing

- Every declared dataset exists with rows > 0 and the expected entity column.
- `landing_required_cols` are present; `landing_forbidden_cols` are absent.
- If `TARGET_HOST_DATASET` is set, the target column is populated.
- Event-level datasets with `has_lifecycle_enrichment=True` emit the declared
  `event_types` and have `event_timestamp` 100% non-null.


In [ ]:
# @cr:code name='landing_inventory' id=probe-l1-inv
_section("1.1 Landing inventory")
rows = []
for name, cfg in DATASETS.items():
    path = f"{RUN_ROOT}/landing/{name}"
    df, err = _try_load(path)
    if err:
        _record("landing", name, "exists", False, err)
        rows.append({"dataset": name, "rows": -1, "cols": -1, "note": err})
        print(f"  FAIL {name}: {err}")
        continue
    n, c = df.count(), len(df.columns)
    entity = _first_col(df, cfg["entity_cols"])
    uniq = df.select(entity).distinct().count() if entity else -1
    _record("landing", name, "exists_nonzero", n > 0 and entity is not None,
            f"rows={n} entity={entity}")
    rows.append({"dataset": name, "rows": n, "cols": c,
                 "entity_col": entity or "-", "unique_entities": uniq, "note": ""})
    mark = "OK" if n > 0 and entity else "FAIL"
    print(f"  [{mark}] {name}: rows={n:,} cols={c} entity={entity} unique={uniq:,}")
_dump(rows, "landing_inventory")


In [ ]:
# @cr:code name='landing_schema' id=probe-l1-sch
_section("1.2 Landing schema — required present, forbidden absent")
rows = []
for name, cfg in DATASETS.items():
    df, err = _try_load(f"{RUN_ROOT}/landing/{name}")
    if err:
        continue
    required = cfg.get("landing_required_cols", [])
    forbidden = cfg.get("landing_forbidden_cols", [])
    cols_lower = {c.lower() for c in df.columns}
    missing_required = [c for c in required if c.lower() not in cols_lower]
    present_forbidden = [c for c in forbidden if c.lower() in cols_lower]
    _record("landing", name, "required_cols_present", not missing_required,
            f"missing={missing_required}")
    _record("landing", name, "forbidden_cols_absent", not present_forbidden,
            f"present={present_forbidden}")
    rows.append({"dataset": name, "missing_required": ",".join(missing_required),
                 "present_forbidden": ",".join(present_forbidden)})
    mark = "OK" if not missing_required and not present_forbidden else "FAIL"
    print(f"  [{mark}] {name}: missing={missing_required or '-'} forbidden_hits={present_forbidden or '-'}")
_dump(rows, "landing_schema")


In [ ]:
# @cr:code name='landing_target' id=probe-l1-tgt
_section("1.3 Landing target enrichment")
if TARGET_HOST_DATASET is None:
    print("  (no TARGET_HOST_DATASET configured — skipping)")
else:
    df, err = _try_load(f"{RUN_ROOT}/landing/{TARGET_HOST_DATASET}")
    if err:
        _record("landing", TARGET_HOST_DATASET, "target_host_loads", False, err)
    else:
        present = TARGET_COLUMN in df.columns
        _record("landing", TARGET_HOST_DATASET, f"{TARGET_COLUMN}_present", present, "")
        print(f"  [{'OK' if present else 'FAIL'}] {TARGET_COLUMN} present on {TARGET_HOST_DATASET}")
        if present:
            display_table(df.groupBy(TARGET_COLUMN).count().toPandas())


In [ ]:
# @cr:code name='landing_lifecycle' id=probe-l1-lc
_section("1.4 Landing lifecycle enrichment (event_type / event_timestamp)")
rows = []
for name, cfg in DATASETS.items():
    if not cfg.get("has_lifecycle_enrichment"):
        continue
    df, err = _try_load(f"{RUN_ROOT}/landing/{name}")
    if err:
        continue
    has_et = "event_type" in df.columns
    has_ts = "event_timestamp" in df.columns
    _record("landing", name, "lifecycle_enriched", has_et and has_ts, "")
    if has_et:
        observed = {r["event_type"] for r in df.select("event_type").distinct().collect()}
        expected = set(cfg.get("event_types", []))
        _record("landing", name, "event_type_values_ok", observed <= expected or observed == expected,
                f"observed={sorted(observed)}")
        rows.append({"dataset": name, "event_types": ",".join(sorted(observed))})
        print(f"  {name}: event_types={sorted(observed)}")
    if has_ts:
        nulls = df.filter(F.col("event_timestamp").isNull()).count()
        _record("landing", name, "event_timestamp_nonnull", nulls == 0, f"nulls={nulls}")
        if nulls:
            print(f"  [FAIL] {name}: event_timestamp has {nulls} NULLs")
_dump(rows, "landing_lifecycle")


[//]: # (cr:doc name='layer_2_bronze' id=probe-l2-md)
## Layer 2 — Bronze

- Every event dataset has its `bronze_path` populated.
- Universal aggregation family present: `event_count_*`, `days_since_last_event`,
  `active_span_days`, `recency_ratio`.
- Datasets with `per_grid_date_mode=True` and configured `event_types` emit
  `event_type_<value>_count_{window}` for every `window` in `BRONZE_WINDOWS`.
- Datasets with a declared `value_column` emit windowed sum/avg/min/max aggregates.
- Expected milestone-delay columns from `EXPECTED_MILESTONE_COLUMNS` are present.


In [ ]:
# @cr:code name='bronze_inventory' id=probe-l2-inv
_section("2.1 Bronze inventory")
rows = []
for name, cfg in DATASETS.items():
    path = f"{RUN_ROOT}/{cfg['bronze_path']}"
    df, err = _try_load(path)
    if err:
        passed = cfg["kind"] == "entity"
        _record("bronze", name, "exists", passed, "entity passthrough" if passed else err)
        rows.append({"dataset": name, "rows": 0, "cols": 0, "note": "entity passthrough" if passed else err})
        print(f"  [{'INFO' if passed else 'FAIL'}] {name}: {'entity passthrough' if passed else err}")
        continue
    n, c = df.count(), len(df.columns)
    _record("bronze", name, "exists_nonzero", n > 0, f"rows={n}")
    rows.append({"dataset": name, "rows": n, "cols": c, "note": ""})
    print(f"  [OK] {name}: rows={n:,} cols={c}")
_dump(rows, "bronze_inventory")


In [ ]:
# @cr:code name='bronze_families' id=probe-l2-fam
_section("2.2 Bronze universal aggregation families")
patterns = ["event_count_", "days_since_last_event", "active_span_days", "recency_ratio"]
rows = []
for name, cfg in DATASETS.items():
    if cfg["kind"] != "event":
        continue
    df, err = _try_load(f"{RUN_ROOT}/{cfg['bronze_path']}")
    if err:
        continue
    for pat in patterns:
        matches = [c for c in df.columns if pat in c]
        _record("bronze", name, f"family_{pat.rstrip('_')}", bool(matches), f"n={len(matches)}")
        rows.append({"dataset": name, "family": pat, "n_matches": len(matches)})
    mark = "OK" if all(any(p in c for c in df.columns) for p in patterns) else "FAIL"
    print(f"  [{mark}] {name}: families {[p.rstrip('_') for p in patterns]}")
_dump(rows, "bronze_families")


In [ ]:
# @cr:code name='bronze_event_type' id=probe-l2-et
_section("2.3 Bronze per-grid-date event_type counts")
rows = []
for name, cfg in DATASETS.items():
    if not cfg.get("per_grid_date_mode"):
        continue
    df, err = _try_load(f"{RUN_ROOT}/{cfg['bronze_path']}")
    if err:
        continue
    all_ok = True
    for et in cfg.get("event_types", []):
        for w in BRONZE_WINDOWS:
            matches = [c for c in df.columns if (f"_{et}_count_" in c or f"event_type_{et}" in c) and c.endswith(f"_{w}")]
            ok = bool(matches)
            all_ok = all_ok and ok
            rows.append({"dataset": name, "event_type": et, "window": w, "n_matches": len(matches),
                         "status": "PASS" if ok else "FAIL"})
    _record("bronze", name, "per_grid_date_event_type_counts", all_ok,
            f"{len(BRONZE_WINDOWS)} windows x {len(cfg.get('event_types', []))} types")
    print(f"  [{'OK' if all_ok else 'FAIL'}] {name} per-grid-date event_type coverage")
_dump(rows, "bronze_event_type_counts")


In [ ]:
# @cr:code name='bronze_value_column' id=probe-l2-vc
_section("2.4 Bronze VALUE_COLUMN windowed aggregates")
rows = []
for name, cfg in DATASETS.items():
    vc = cfg.get("value_column")
    if cfg["kind"] != "event" or not vc or vc == "_event_count":
        continue
    df, err = _try_load(f"{RUN_ROOT}/{cfg['bronze_path']}")
    if err:
        continue
    aggs = {agg: [c for c in df.columns if c.startswith(vc) and f"_{agg}_" in c] for agg in ("sum", "avg", "mean", "min", "max")}
    any_ok = any(v for v in aggs.values())
    _record("bronze", name, f"value_column_aggs({vc})", any_ok,
            " ".join(f"{k}={len(v)}" for k, v in aggs.items()))
    rows.append({"dataset": name, "value_column": vc, **{k: len(v) for k, v in aggs.items()}})
    print(f"  [{'OK' if any_ok else 'FAIL'}] {name}.{vc}: " + " ".join(f"{k}={len(v)}" for k, v in aggs.items()))
_dump(rows, "bronze_value_column")


In [ ]:
# @cr:code name='bronze_milestones' id=probe-l2-ms
_section("2.5 Bronze milestone-delay aggregates")
rows = []
for name, milestones in EXPECTED_MILESTONE_COLUMNS.items():
    cfg = DATASETS.get(name)
    if cfg is None:
        continue
    df, err = _try_load(f"{RUN_ROOT}/{cfg['bronze_path']}")
    if err:
        continue
    for ms in milestones:
        matches = [c for c in df.columns if ms in c.lower()]
        _record("bronze", name, f"milestone_{ms}", bool(matches), f"n={len(matches)}")
        rows.append({"dataset": name, "milestone": ms, "n_matches": len(matches)})
        print(f"  [{'OK' if matches else 'FAIL'}] {name}.{ms}: n={len(matches)}")
_dump(rows, "bronze_milestones")


[//]: # (cr:doc name='layer_3_silver' id=probe-l3-md)
## Layer 3 — Silver

- `silver_merged` exists with rows > 0.
- Each non-spine merged dataset contributes ≥ 1 column to silver. Coverage
  is computed via `classify_silver_columns` (stages/temporal/temporal_merger.py)
  against each dataset's bronze column set — the merger applies the
  `<dataset>__` prefix only on name collisions, so a naive `startswith`
  check mis-attributes the majority of features.
- `FORBIDDEN_SILVER_SUBSTRINGS` are absent from the column set.
- Target distribution is sane (non-null ratio, class balance).


In [ ]:
# @cr:code name='silver_inventory' id=probe-l3-inv
_section("3.1 Silver inventory & target")
silver, err = _try_load(f"{RUN_ROOT}/silver/silver_merged")
if err:
    _record("silver", "silver_merged", "exists", False, err)
    print(f"  [FAIL] silver_merged: {err}")
else:
    n, c = silver.count(), len(silver.columns)
    entity = _first_col(silver, ["entity_id", "ACCOUNT_ID"])
    date = _first_col(silver, ["as_of_date", "feature_timestamp"])
    _record("silver", "silver_merged", "exists_nonzero", n > 0, f"rows={n} cols={c}")
    print(f"  [OK] rows={n:,} cols={c} entity={entity} time={date}")
    if TARGET_COLUMN in silver.columns:
        nn = silver.filter(F.col(TARGET_COLUMN).isNotNull()).count()
        _record("silver", "silver_merged", f"{TARGET_COLUMN}_labelled_present", nn > 0, f"labelled={nn}")
        print(f"  {TARGET_COLUMN} non-null: {nn:,} / {n:,} ({100*nn/n:.2f}%)")


In [ ]:
# @cr:code name='silver_column_origin' id=probe-l3-pfx
_section("3.2 Silver column-origin coverage (merger-contract-aware)")
from customer_retention.stages.temporal.temporal_merger import classify_silver_columns

silver, err = _try_load(f"{RUN_ROOT}/silver/silver_merged")
if err:
    _record("silver", "silver_merged", "silver_origin_coverage", False, err)
    print(f"  [FAIL] {err}")
else:
    bronze_cols = {}
    for name, cfg in DATASETS.items():
        bp = cfg.get("bronze_path")
        src = f"{RUN_ROOT}/landing/{name}" if cfg["kind"] == "entity" else f"{RUN_ROOT}/{bp}"
        bdf, berr = _try_load(src)
        if berr:
            bronze_cols[name] = []
            print(f"  [WARN] bronze/{name} missing ({berr[:80]}) — treated as empty")
        else:
            bronze_cols[name] = list(bdf.columns)

    base_source = next((n for n, c in DATASETS.items() if c.get("silver_prefix") is None and c.get("kind") == "entity"), None)
    if base_source is None:
        _record("silver", "silver_merged", "silver_origin_coverage", False, "no base_source declared (silver_prefix=None)")
        print("  [FAIL] no dataset has silver_prefix=None — cannot identify base source")
    else:
        spine = tuple(c for c in ("entity_id", "as_of_date") if c in silver.columns)
        try:
            origin = classify_silver_columns(list(silver.columns), bronze_cols, base_source=base_source, spine_columns=spine)
            _record("silver", "silver_merged", "silver_origin_no_ambiguous_columns", True, f"classified={len(origin)}")
        except ValueError as e:
            origin = {}
            _record("silver", "silver_merged", "silver_origin_no_ambiguous_columns", False, str(e)[:200])
            print(f"  [FAIL] {e}")

        contributions = {}
        for col, src in origin.items():
            if src == "spine":
                continue
            contributions[src] = contributions.get(src, 0) + 1

        rows = []
        for name in DATASETS:
            if name == base_source:
                continue
            n_cols = contributions.get(name, 0)
            _record("silver", name, "silver_origin_contribution", n_cols >= 1, f"n={n_cols}")
            rows.append({"dataset": name, "n_columns": n_cols})
            print(f"  [{'OK' if n_cols else 'FAIL'}] {name}: n_columns={n_cols}")
        _dump(rows, "silver_column_origin")


In [ ]:
# @cr:code name='silver_forbidden' id=probe-l3-fbd
_section("3.3 Silver anti-leakage forbidden substrings")
silver, err = _try_load(f"{RUN_ROOT}/silver/silver_merged")
if err:
    print(f"  [FAIL] {err}")
else:
    cols_lower = [c.lower() for c in silver.columns]
    hits = []
    for needle in FORBIDDEN_SILVER_SUBSTRINGS:
        for c, cl in zip(silver.columns, cols_lower):
            if needle.lower() in cl:
                hits.append({"substring": needle, "column": c})
    _record("silver", "silver_merged", "no_forbidden_columns", not hits, f"hits={len(hits)}")
    if hits:
        print(f"  [FAIL] {len(hits)} forbidden substring hits:")
        for h in hits[:20]:
            print(f"    {h['substring']} -> {h['column']}")
    else:
        print(f"  [OK] {len(FORBIDDEN_SILVER_SUBSTRINGS)} forbidden substrings absent")
    _dump(hits, "silver_forbidden_hits")


[//]: # (cr:doc name='summary' id=probe-summary-md)
## Summary — aggregate PASS/FAIL and emit `result.json`


In [ ]:
# @cr:code name='probe_summary' id=probe-summary
_section("Summary")
total = len(CHECKS)
passed = sum(1 for c in CHECKS if c["status"] == "PASS")
failed = total - passed
print(f"  total={total}  PASS={passed}  FAIL={failed}")

summary_df = native_pd.DataFrame(CHECKS)
if failed:
    display_table(summary_df[summary_df["status"] == "FAIL"])

result = {
    "run_id": _namespace.run_id,
    "status": "PASS" if failed == 0 else "FAIL",
    "total": total, "passed": passed, "failed": failed,
    "checks": CHECKS,
}
result_path = RESULT_DIR / "result.json"
result_path.write_text(json.dumps(result, indent=2, default=str))
print(f"  result -> {result_path}")
